# AG-HYPOPT · experiment_1 · Trial 01

**Hypothesis:** <one line: what this trial tests>

> ▶ = run the cell · ✍️ = write here before continuing · ⛔ never "Run All" | full rules: `instructions.md`


## 1. ✍️ Read and summarize

**State of the campaign.** Registry (`trials.json`) holds only `baseline_17g`:
objective **0.001578 ± 0.000859** (μ rel-RMSE 3.6%, γ rel-RMSE 4.3% on the 14-exp
synthetic benchmark) — the 17f/z-form γ-score config (n_runs=200, n_iter=200,
lr_mu=15, lr_gamma=0.5, σ_ref=10, clip=10, γ_anneal=0.5, h_s_min=0.05). Known
weaknesses from 18c/19-series context: 3nW Trans05 γ outlier (+2.14) and a μ bias
at low transmission; score-design failures dominate, hyperparameters are secondary.

**What this trial should do.** This is trial_01 — the campaign's first real step.
With only one completed trial the proposer cold-starts (LHS spread), so the batch
is exploratory by design: it maps the space, it does not exploit yet. My job is to
pick ONE config that (a) is a sensible first probe — physically motivated, not
extreme — and (b) gives TPE a useful second point regardless of outcome.

**Baseline to beat:** 0.001578. Any trial with objective < baseline and sane
breakdown advances the campaign.


In [1]:
# 2. ▶ Propose candidate trials (AGHyperopt)
# API contract: AGHyperopt is implemented in ag_hypopt.py. fit() reads the space and the trial
# history; propose_trials() prints a candidate table (ID | EI | explore | params) and returns
# the candidates for the cells below.
import os

EXPERIMENT_DIR = os.getcwd()                     # notebooks run in place from the experiment folder
SPACE_PATH = os.path.join(EXPERIMENT_DIR, 'space.json')
TRIALS_PATH = os.path.join(EXPERIMENT_DIR, 'trials.json')

MAX_TRIALS = 10    # campaign cap: stop generating new trials after this many (None = unlimited)

from ag_hypopt import AGHyperopt

opt = AGHyperopt()
opt.fit(SPACE_PATH, TRIALS_PATH)
proposed_trials = opt.propose_trials(10)


ID |      EI | slot     | params
 1 | 0.5955 |          | {"n_runs": 151, "n_iter": 235, "lr_mu": 17.977930848140343, "lr_gamma": 1.3974709843880426, "sigma_ref": 17.877302401613292, "clip": 17.341424199062452, "gamma_anneal": 0.3325606491204983, "h_s_min": 0.04544774435695538}
 2 | 0.6369 |          | {"n_runs": 156, "n_iter": 160, "lr_mu": 5.257679441285193, "lr_gamma": 1.2016941285029938, "sigma_ref": 18.297017131840644, "clip": 15.577480679395027, "gamma_anneal": 0.5855467732664759, "h_s_min": 0.09178315510766799}
 3 | 0.6759 |          | {"n_runs": 156, "n_iter": 134, "lr_mu": 28.39410366266651, "lr_gamma": 0.7595346886003854, "sigma_ref": 16.304722129623777, "clip": 16.474982861240385, "gamma_anneal": 0.4760387400004431, "h_s_min": 0.11071588013159916}
 4 | 0.6881 |          | {"n_runs": 222, "n_iter": 109, "lr_mu": 20.285108623132682, "lr_gamma": 0.40041854194734083, "sigma_ref": 13.170572874492724, "clip": 17.801046099022493, "gamma_anneal": 0.17545461439900556, "h_s_min": 0.01

## 3. ✍️ Analyze and choose

**Cold-start batch (LHS — EI order carries no physics signal).** Screening each
candidate against the 17f baseline (n_runs=200, n_iter=200, lr_mu=15, lr_gamma=0.5,
σ_ref=10, clip=10, γ_anneal=0.5, h_s_min=0.05) with the physics of the two
channels in mind:

- **μ travel** ∝ lr_mu·n_iter·(10/σ_ref)²  — the REINFORCE μ-score scales as
  1/σ_ref², so a larger σ_ref silently starves μ. Baseline travel = 3000 (a.u.),
  sufficient for μ rel-RMSE 3.6%. Candidates 1,2,3,4,7,10 sit at 0.08–0.48× that
  → **μ would stall in the low basin; a loss there would be an under-convergence
  artefact, not a parameter signal — confounded data for TPE. Rejected.**
- **γ drive** ∝ lr_gamma·n_iter·(1−γ_anneal/2): baseline ≈ 75. End-stage γ lr
  (anneal) matters — the baseline's residual is a 3nW T05 γ overshoot (+2.14),
  so very hot late-γ candidates (1, 9) risk amplifying that outlier.
- Budget n_runs·n_iter ≤ 40k ≈ 3.5 h; cheap configs are *fast trials*, not
  wasted ones — runtime scales with the budget.

**Choice: candidate 5** — n_runs=109, n_iter=127 (budget 13.8k ⇒ ~1.2 h),
lr_mu=30.3 (2×), lr_gamma=0.75 (1.5×), **σ_ref=8.2** (score-scale probe DOWN),
γ_anneal=0.11 (mild ⇒ sustained late-γ drive), h_s_min=0.139 (σ-channel floor up).
μ travel 1.9× baseline (no convergence confound), γ drive 1.2×.

*Hypothesis:* (a) the dominant residual is μ bias in the low basin → 2× μ drive
at a *smaller* σ_ref (stronger, still self-normalized REINFORCE gradient) should
pull μ̂ closer to truth; (b) a mild γ anneal keeps γ correcting at the end —
tests whether the 3nW T05 overshoot is late-stage drive (would worsen) or a
settled attractor (unchanged); (c) h_s_min up blunts the σ channel — expect γ
guidance to degrade if the σ-dimension bandwidth floor binds.

*Confirm:* objective < 0.001578 with μ rel-RMSE < 3.6% and no new γ outlier.
*Refute:* objective ≥ baseline, or μ oscillating/overshooting (rel-RMSE worse).


In [2]:
# 4. ▶ Run the chosen trial   ⛔ ~3.5 h: do not interrupt unless obviously broken
INDEX = 5                # <-- your chosen candidate (1-based, from the table in cell 2)
TRIAL_ID = 'trial_01'   # auto-stamped at generation; do not edit

assert 1 <= INDEX <= len(proposed_trials), 'bad INDEX'
CHOSEN = proposed_trials[INDEX - 1]['params']
print('chosen:', CHOSEN)

# API contract: run_trial / compute_objective / format_report come from ag_hypopt (next session).
from ag_hypopt import run_trial, compute_objective, format_report

import traceback, time
t0 = time.time()
try:
    results = run_trial(CHOSEN)
    print(f'trial finished in {(time.time()-t0)/60:.1f} min')
    objective, uncertainty, breakdown = compute_objective(results)
    print(f'objective (MSE vs true) = {objective:.4f} ± {uncertainty:.4f}')
    print(format_report(breakdown))
except Exception:
    traceback.print_exc()
    results, objective, uncertainty = None, None, None


chosen: {'n_runs': 109, 'n_iter': 127, 'lr_mu': 30.28257727087576, 'lr_gamma': 0.7466281223519423, 'sigma_ref': 8.225435580672036, 'clip': 12.515671626550454, 'gamma_anneal': 0.11423407703487631, 'h_s_min': 0.13926407501554722}
Running  1nW Trans05 ... 

μ 9.39 -> 11.45 | γ 8.5 -> 9.27 | NLL 1.79


Running  1nW Trans10 ... 

μ 12.37 -> 13.52 | γ 8.5 -> 8.25 | NLL 1.95


Running  1nW Trans20 ... 

...

Running  3nW Trans80 ... 

μ 137.54 -> 144.10 | γ 14.1 -> 14.16 | NLL 2.21


Running 3nW Trans100 ... 

μ 175.71 -> 187.48 | γ 14.1 -> 14.37 | NLL 2.12



Total: 69.9 min
trial finished in 69.9 min
objective (MSE vs true) = 0.0049 ± 0.0021
exp            μ_true  μ_final       Δμ | γ_true  γ_final       Δγ
1nW Trans05  ...
1nW Trans10  ...
1nW Trans20  ...
1nW Trans40  ...
1nW Trans60  ...
1nW Trans80  ...
1nW Trans100 ...
3nW Trans05  ...
3nW Trans10  ...
3nW Trans20  ...
3nW Trans40  ...
3nW Trans60  ...
3nW Trans80  ...
3nW Trans100 ...
μ: RMSE 4.146 (rel 8.6%) | γ: RMSE 0.630 (rel 5.0%)


## 5. ✍️ Analyze the results, write analysis and summary

**What happened.** Ran the full 14-exp benchmark in 69.9 min (budget 13.8k, as predicted).
Objective **0.0049 ± 0.0021** — ≈ **3.1× worse** than the baseline 0.001578 ± 0.000859.
μ rel-RMSE 8.6% (baseline 3.6%), γ rel-RMSE 5.0% (baseline 4.3%). The cell-3 hypothesis is
**refuted on every branch** — and the refutation is informative.

**μ channel — the 2× drive attack backfired.** Hypothesis (a) said the low-basin μ residual
is under-convergence starvation, so 2× lr_mu at a *smaller* σ_ref (stronger self-normalized
REINFORCE gradient) should pull μ̂ closer to truth. Instead μ overshot on **all 14 exps**
(Δμ = +0.31 … +11.77, every single one positive): rel-RMSE 8.6% vs 3.6%. The pattern is a
pure sign-coherent bias amplified by drive, worst in relative terms exactly where hypothesis
(a) predicted improvement — the low-transmission basin (1nW T05 +21.9%, 3nW T05 +17.3%) —
and growing in absolute terms toward high trans (3nW T100 +11.77). More drive did not cure
the bias; it fed it. That is the signature of a score/estimator bias, not lr starvation.

**γ channel — mild-anneal probe: the T05 outlier is a settled attractor.** Hypothesis (b)
asked whether the known 3nW T05 γ overshoot (+2.14 at baseline) is late-stage drive (mild
anneal + 1.2× drive ⇒ should worsen) or a settled attractor (unchanged). Result: γ 14.1 →
16.26, overshoot **+2.16** — essentially identical. The other 13 γ errors stayed small
(−0.25 … +0.27; excluding T05, γ rel-RMSE ≈ 1.3%). Verdict: the outlier is not driven by
late-stage γ drive; it is a fixed point of the γ-score landscape.

**σ-channel floor (c).** h_s_min 0.139 (baseline 0.05) degraded γ only mildly overall (5.0%
vs 4.3%), and that degradation is almost entirely the unchanged T05 outlier — the floor did
not visibly bind on the other 13 exps. Effect: secondary, not cleanly separable.

**Per-exp detail** (from the run log; format_report prints rows as `...` by design):

| exp | μ_true→μ_final | Δμ (rel) | γ_true→γ_final | Δγ |
|---|---|---|---|---|
| 1nW Trans05 | 9.39→11.45 | +2.06 (+21.9%) | 8.5→9.27 | +0.77 |
| 1nW Trans10 | 12.37→13.52 | +1.15 (+9.3%) | 8.5→8.25 | −0.25 |
| 1nW Trans20 | 17.32→17.63 | +0.31 (+1.8%) | 8.5→8.33 | −0.17 |
| 1nW Trans40 | 38.41→39.20 | +0.79 (+2.1%) | 8.5→8.73 | +0.23 |
| 1nW Trans60 | 61.37→63.41 | +2.04 (+3.3%) | 8.5→8.69 | +0.19 |
| 1nW Trans80 | 79.36→82.86 | +3.50 (+4.4%) | 8.5→8.60 | +0.10 |
| 1nW Trans100 | 70.82→74.52 | +3.70 (+5.2%) | 8.5→8.66 | +0.16 |
| 3nW Trans05 | 13.20→15.49 | +2.29 (+17.3%) | 14.1→16.26 | **+2.16** |
| 3nW Trans10 | 24.48→25.14 | +0.66 (+2.7%) | 14.1→14.10 | +0.00 |
| 3nW Trans20 | 34.28→34.73 | +0.45 (+1.3%) | 14.1→14.18 | +0.08 |
| 3nW Trans40 | 84.89→87.38 | +2.49 (+2.9%) | 14.1→14.14 | +0.04 |
| 3nW Trans60 | 103.20→106.47 | +3.27 (+3.2%) | 14.1→14.18 | +0.08 |
| 3nW Trans80 | 137.54→144.10 | +6.56 (+4.8%) | 14.1→14.16 | +0.06 |
| 3nW Trans100 | 175.71→187.48 | +11.77 (+6.7%) | 14.1→14.37 | +0.27 |

**What was learned.** (1) The μ residual is a positive, drive-amplified bias — a genuinely
useful negative result: it removes convergence starvation from the candidate causes and
points at the μ-score's self-normalization as the suspect. (2) 3nW T05's γ overshoot is
robust to both stronger drive and a milder anneal — a stable feature, so tuning γ lr /
anneal further is unlikely to move it. (3) This trial was still a good TPE second point:
it maps the lr/σ_ref direction as *steeply harmful*, which cold LHS could not know.

*Structural note for Anuar (no action taken):* the sign-coherent μ overshoot under
stronger drive and the T05 γ fixed point both look like score/estimator properties rather
than hp-tunable behaviour; a future structural pass might examine μ-score normalization
and the T05 γ landscape. Out of scope here — hyperparameter campaign continues.

**Summary:** trial_01 (2× lr_mu, σ_ref 8.2, mild γ-anneal, h_s_min up) ran the 14-exp benchmark in 70 min: objective 0.0049 ± 0.0021 — 3.1× worse than baseline, driven by sign-coherent μ overshoot on all 14 exps (rel-RMSE 8.6% vs 3.6%); 3nW T05 γ overshoot unchanged at +2.16.
**Key insight:** μ overshoot that grows with drive plus a T05 γ overshoot that ignores both drive and anneal refute convergence starvation — these residuals are score/estimator artifacts, not hyperparameters to tune away.

In [3]:
# 6. ▶ Record the trial in trials.json  (fill the two strings below first; copy them from cell 5)
import json

SUMMARY = 'trial_01 (2× lr_mu, σ_ref 8.2, mild γ-anneal, h_s_min up) ran the 14-exp benchmark in 70 min: objective 0.0049 ± 0.0021 — 3.1× worse than baseline, driven by sign-coherent μ overshoot on all 14 exps (rel-RMSE 8.6% vs 3.6%); 3nW T05 γ overshoot unchanged at +2.16.'
KEY_INSIGHT = 'μ overshoot that grows with drive plus a T05 γ overshoot that ignores both drive and anneal refute convergence starvation — these residuals are score/estimator artifacts, not hyperparameters to tune away.'

entry = {
    'trial_id': TRIAL_ID,
    'config': CHOSEN,
    'objective': objective,
    'uncertainty': uncertainty,
    'summary': SUMMARY,
    'key_insight': KEY_INSIGHT,
    'notebook': f'{TRIAL_ID}.ipynb',
}
data = json.load(open(TRIALS_PATH))
assert not any(t.get('trial_id') == TRIAL_ID for t in data['trials']),     f'{TRIAL_ID} is already registered in trials.json'
data['trials'].append(entry)
json.dump(data, open(TRIALS_PATH, 'w'), indent=2)
best = min((t for t in data['trials'] if t.get('objective') is not None),
           key=lambda t: t['objective'], default=None)
print('saved', TRIAL_ID, '| current best:', best['trial_id'] if best else None)


saved trial_01 | current best: baseline_17g


In [4]:
# 7. ▶ Generate the next trial (or end the campaign)
import os, re, json as _json

num = int(re.search(r'(\d+)$', TRIAL_ID).group(1))
nxt = num + 1
if MAX_TRIALS is not None and nxt > MAX_TRIALS:
    print(f'Campaign complete: cap MAX_TRIALS={MAX_TRIALS} reached after {TRIAL_ID}. Stop here.')
else:
    template_path = os.path.join(EXPERIMENT_DIR, 'template.ipynb')
    target = os.path.join(EXPERIMENT_DIR, f'trial_{nxt:02d}.ipynb')
    assert not os.path.exists(target), f'{target} already exists: refusing to overwrite'

    nb = _json.load(open(template_path))

    def _stamp(old, new):
        for c in nb['cells']:
            if old in ''.join(c.get('source', [])):
                c['source'] = [s.replace(old, new) for s in c['source']]
                return True
        raise RuntimeError(f'stamp target {old!r} not found in the template')

    _stamp('{N}', f'{nxt:02d}')
    _stamp('trial_XXX', f'trial_{nxt:02d}')
    _json.dump(nb, open(target, 'w'), indent=1)
    print(f'Created {os.path.basename(target)}. Open it and follow its cells from the top.')


Created trial_02.ipynb. Open it and follow its cells from the top.
